# Большой прогон: внешность и трекер на всех кадрах

Ничего в проекте он не меняет: каждый этап пишет свою папку и свой архив на Диске. Решение о переносе принимается после сравнения на ноутбуке.

1. **Runtime → Change runtime type → A100 GPU**, если её нет, то L4. На T4 успеют только первые этапы. Другие среды остановите: **Runtime → Disconnect and delete runtime**.
2. На Диск в папку `mct` положите новый `mct_code.zip` (лежит в проекте: `artifacts/mct_code.zip`) рядом со старым `mct_colab.zip`. Старый заново загружать не нужно.
3. Для отдельного опыта выполните ячейки подключения, установки, проверки GPU и функций, затем нужный этап. **Run all** запускает также долгие опыты на всех кадрах.
4. На Диске нужно около 3 ГБ свободного места.
5. Если сессия оборвётся, запустите всё снова: готовое будет пропущено.
6. После каждого этапа на Диске появляется `mct/big_<этап>.zip`. Готовые архивы можно скачивать сразу, не дожидаясь конца: положите их в папку проекта `artifacts/trial/big/`.

Скачивается по ходу работы: веса `yolo11x-seg.pt` с GitHub Ultralytics, `facebook/dinov2-large` с Hugging Face, OSNet-AIN через BoxMOT.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls -la /content/drive/MyDrive/mct/mct_colab.zip /content/drive/MyDrive/mct/mct_code.zip

In [ ]:
%%bash
set -e
mkdir -p /content/mct
unzip -oq /content/drive/MyDrive/mct/mct_colab.zip -d /content/mct
unzip -oq /content/drive/MyDrive/mct/mct_code.zip -d /content/mct
pip install -q boxmot==26.0.0 ultralytics==8.4.203 imageio-ffmpeg lap pycocotools pyarrow transformers
ls /content/mct /content/mct/data /content/mct/artifacts/adapt

In [ ]:
import torch, boxmot, ultralytics
print('cuda', torch.cuda.is_available(), 'boxmot', boxmot.__version__, 'ultralytics', ultralytics.__version__)
assert torch.cuda.is_available(), 'GPU не подключён: смените среду выполнения'

In [ ]:
import os, shutil, subprocess, sys, time, traceback
from pathlib import Path
import torch

ROOT = Path('/content/mct')
sys.path.insert(0, str(ROOT / 'src'))
from mct.perception import valid_embeddings, valid_perception_output
DRIVE = Path('/content/drive/MyDrive/mct')
BIG = DRIVE / 'big'
BIG.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, 'PYTHONPATH': 'src'}
assert torch.cuda.is_available(), 'Для детекции нужен GPU Colab'
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
PARALLEL = 2 if GPU_GB >= 40 else 1
VIDEOS = sorted(ROOT.glob('data/*.mkv'))
EXPECTED = {f'camera_new_{i}' for i in range(1, 6)}
assert {video.stem for video in VIDEOS} == EXPECTED, f'Нужны пять исходных камер: найдено {[video.stem for video in VIDEOS]}'
print('GPU:', round(GPU_GB, 1), 'ГБ; камер одновременно:', PARALLEL, 'записей:', len(VIDEOS))
for video in VIDEOS:
    folder = ROOT / 'one' / video.stem
    folder.mkdir(parents=True, exist_ok=True)
    if not (folder / video.name).exists():
        (folder / video.name).symlink_to(video)

def last_line(path):
    lines = [line for line in path.read_text(errors='ignore').splitlines() if 'кадров' in line or 'снимков' in line or 'Error' in line or 'Traceback' in line]
    return lines[-1][-50:] if lines else 'запуск'

def by_camera(title, command, is_done):
    # command(папка с одной записью, имя камеры) -> аргументы python; is_done(имя камеры) -> готово ли
    waiting = [video.stem for video in VIDEOS if not is_done(video.stem)]
    running, failed = {}, []
    print('===', title, 'осталось камер:', len(waiting), flush=True)
    while waiting or running:
        while waiting and len(running) < PARALLEL:
            camera = waiting.pop(0)
            log = Path(f'/content/{title}_{camera}.log')
            stream = log.open('a')
            running[camera] = (subprocess.Popen(['python', *map(str, command(ROOT / 'one' / camera, camera))], cwd=ROOT, env=ENV, stdout=stream, stderr=subprocess.STDOUT), log, stream)
        time.sleep(30)
        print(time.strftime('%H:%M'), {name: last_line(log) for name, (_, log, _) in running.items()}, flush=True)
        for name, (job, log, stream) in list(running.items()):
            if job.poll() is not None:
                stream.close()
                print(name, 'закончила, код', job.returncode, flush=True)
                if job.returncode:
                    failed.append(name)
                    print(log.read_text(errors='ignore')[-1500:], flush=True)
                del running[name]
    missing = [video.stem for video in VIDEOS if not is_done(video.stem)]
    print(title, 'ошибки:', failed or 'нет', 'неполные результаты:', missing or 'нет', flush=True)
    return not failed and not missing

def pack(stage):
    temporary = DRIVE / f'.big_{stage}.partial.zip'
    archive = shutil.make_archive(str(temporary.with_suffix('')), 'zip', BIG, stage)
    final = DRIVE / f'big_{stage}.zip'
    os.replace(archive, final)
    print('Архив:', final, round(final.stat().st_size / 1e6), 'МБ', flush=True)

def stage(name, job):
    # Ошибка одного этапа не останавливает следующие
    started = time.time()
    try:
        job()
        pack(name)
    except Exception:
        traceback.print_exc()
    print('=== этап', name, 'занял', round((time.time() - started) / 60), 'мин', flush=True)

def reembed(perception, out, *options, suffix='x', model='osnet_ain_x1_0_msmt17'):
    out.mkdir(parents=True, exist_ok=True)
    return by_camera(out.name + '_' + suffix, lambda folder, camera: ['scripts/reembed.py', folder, '--perception', perception, '--out', out, '--suffix', suffix, '--kinds', 'box', '--device', '0', '--models', model, *options],
              lambda camera: valid_embeddings(out / f'{camera}_emb_{suffix}.npz'))

def detection(name, model, imgsz):
    out = BIG / name
    out.mkdir(parents=True, exist_ok=True)
    subprocess.run(['python', '-c', f"from ultralytics import YOLO; YOLO('{model}'); from mct.embedder import create_embedder; create_embedder('0')"], cwd=ROOT, env=ENV, check=True)
    return out, lambda *options: by_camera(name, lambda folder, camera: ['scripts/run_perception.py', folder, '--out', out, '--device', '0', '--model', model, '--imgsz', imgsz, *options],
                                           lambda camera: valid_perception_output(out, camera, {'model': model, 'imgsz': imgsz, 'every_frame': '--every-frame' in options}))

def full_pass(name, model, imgsz, *options):
    def job():
        out, run = detection(name, model, imgsz)
        if not run(*options):
            raise RuntimeError(f'{name}: детекция одной или нескольких камер не завершилась')
        if not reembed(out, out, '--every', '3', suffix='dense'):
            raise RuntimeError(f'{name}: извлечение признаков одной или нескольких камер не завершилось')
    stage(name, job)

## 1. Внешность на готовой детекции

Сейчас внешность решает почти всё в счётчике: без неё 91 трек, с ней 61. Проверяем снимки каждые 0.1 с вместо 0.3 с и вырезки из исходного кадра 2592×1944 вместо 1296×972: снимки реже считаются из этих же файлов. Заодно универсальная модель DINOv2.

In [ ]:
PERCEPTION = ROOT / 'artifacts/perception'
def appearance():
    if not reembed(PERCEPTION, BIG / 'reid' / 'every1', '--every', '1'):
        raise RuntimeError('every1: не все камеры обработаны')
    if not reembed(PERCEPTION, BIG / 'reid' / 'every1_full', '--every', '1', '--full-res'):
        raise RuntimeError('every1_full: не все камеры обработаны')
    if not reembed(PERCEPTION, BIG / 'reid' / 'dino_every3_full', '--every', '3', '--full-res', model='facebook/dinov2-large'):
        raise RuntimeError('dino_every3_full: не все камеры обработаны')
stage('reid', appearance)

## 2. Дообучение модели внешности на самой записи

Подписи без ручной разметки: две камеры долго видели человека в одном месте значит свои, двое видны одновременно и далеко друг от друга значит чужие. Размеченная толпа не участвует, а модели половин записи проверяются на другой половине.

In [ ]:
def adaptation():
    job = subprocess.Popen(['python', 'scripts/adapt_reid.py', 'data', '--perception', 'artifacts/perception', '--labels', 'artifacts/adapt', '--out', str(BIG / 'adapt'), '--device', '0'], cwd=ROOT, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='ignore')
    for line in job.stdout:
        if not line.startswith('INFO') and not line.startswith(' '):
            print(line, end='', flush=True)
    assert job.wait() == 0, 'дообучение упало'
stage('adapt', adaptation)

## 3. Трекер на всех кадрах

Сейчас трекер видит 10 кадров в секунду, хотя его модель движения рассчитана на 25–30. Детектор и все пороги те же, в таблицы идут те же кадры, поэтому дальше конвейер не меняется. Проверяем, станет ли меньше перескоков и разрывов.

In [ ]:
full_pass('frames_m960', 'yolo11m-seg.pt', 960, '--every-frame')

## 4. Повтор основной детекции

Те же настройки, что в сданном результате: проверка повторяемости, которой до сих пор нет. На результат не влияет.

In [ ]:
full_pass('repeat_m960', 'yolo11m-seg.pt', 960)

## 4а. Следующий опыт: YOLO11x на тех же 10 кадрах/с

В предыдущем опыте YOLO11x одновременно поменялись детектор и частота работы трекера, поэтому причина ухудшения неясна. Этот этап меняет только детектор и сохраняет частоту основного варианта. Запускается исключительно на GPU в Colab. Результат появится как `big_x960_sampled.zip`; после скачивания оцените его локально командой `python scripts/eval_detection_run.py <папка_с_результатом>`.

In [ ]:
full_pass('x960_sampled', 'yolo11x-seg.pt', 960)

## 5. Детекторы крупнее, тоже на всех кадрах

Проверяем, делит ли крупная модель слипшихся людей и находит ли дальних. Если время кончилось, эти этапы можно не ждать: готовые архивы уже на Диске.

In [ ]:
full_pass('frames_x960', 'yolo11x-seg.pt', 960, '--every-frame')

In [ ]:
full_pass('frames_m1280', 'yolo11m-seg.pt', 1280, '--every-frame')

In [ ]:
full_pass('frames_x1280', 'yolo11x-seg.pt', 1280, '--every-frame')

In [ ]:
print(sorted(path.name for path in DRIVE.glob('big_*.zip')))
print('ГОТОВО. Скачайте с Диска архивы mct/big_*.zip и положите в папку проекта artifacts/trial/big/.')